In [1]:
from regions import Regions
from regions import CircleSkyRegion, EllipseSkyRegion
from astroquery.svo_fps import SvoFps
from dust_extinction.averages import RL85_MWGC, CT06_MWLoc
import numpy as np
from astropy.io import fits
from astropy.wcs import WCS
from astropy import units as u
from astropy import constants as const

reg_HII = Regions.read('/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/resolved_HII_regions.reg')

imgfile_405410 = '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/f405n_minus_f410m.fits'
imgfile_187182 = '/orange/adamginsburg/w51/TaehwaYoo/jwst_w51/f187n_minus_f182m.fits'
img_for_405410 = fits.open(imgfile_405410)[0]
img_for_187182 = fits.open(imgfile_187182)[0]

image_filenames ={
    "f140m": "/orange/adamginsburg/jwst/w51/F140M/pipeline/jw06151-o001_t001_nircam_clear-f140m-merged_i2d.fits",
    "f150w": "/orange/adamginsburg/jwst/w51/F150W/pipeline/jw06151-o001_t001_nircam_clear-f150w-merged_i2d.fits",
    "f162m": "/orange/adamginsburg/jwst/w51/F162M/pipeline/jw06151-o001_t001_nircam_clear-f162m-merged_i2d.fits",
    "f182m": "/orange/adamginsburg/jwst/w51/F182M/pipeline/jw06151-o001_t001_nircam_clear-f182m-merged_i2d.fits",
    "f187n": "/orange/adamginsburg/jwst/w51/F187N/pipeline/jw06151-o001_t001_nircam_clear-f187n-merged_i2d.fits",
    "f210m": "/orange/adamginsburg/jwst/w51/F210M/pipeline/jw06151-o001_t001_nircam_clear-f210m-merged_i2d.fits",
    "f335m": "/orange/adamginsburg/jwst/w51/F335M/pipeline/jw06151-o001_t001_nircam_clear-f335m-merged_i2d.fits",
    "f360m": "/orange/adamginsburg/jwst/w51/F360M/pipeline/jw06151-o001_t001_nircam_clear-f360m-merged_i2d.fits",
    "f405n": "/orange/adamginsburg/jwst/w51/F405N/pipeline/jw06151-o001_t001_nircam_clear-f405n-merged_i2d.fits",
    "f410m": "/orange/adamginsburg/jwst/w51/F410M/pipeline/jw06151-o001_t001_nircam_clear-f410m-merged_i2d.fits", # weird, the filename is different from what is downloaded with the STScI pipeline...
    "f480m": "/orange/adamginsburg/jwst/w51/F480M/pipeline/jw06151-o001_t001_nircam_clear-f480m-merged_i2d.fits",
    "f560w": "/orange/adamginsburg/jwst/w51/F560W/pipeline/jw06151-o002_t001_miri_f560w_i2d.fits",
    "f770w": "/orange/adamginsburg/jwst/w51/F770W/pipeline/jw06151-o002_t001_miri_f770w_i2d.fits",
    "f1000w": "/orange/adamginsburg/jwst/w51/F1000W/pipeline/jw06151-o002_t001_miri_f1000w_i2d.fits",
    "f1280w": "/orange/adamginsburg/jwst/w51/F1280W/pipeline/jw06151-o002_t001_miri_f1280w_i2d.fits",
    "f2100w": "/orange/adamginsburg/jwst/w51/F2100W/pipeline/jw06151-o002_t001_miri_f2100w_i2d.fits",
    
}
f187n_header = fits.getheader(image_filenames['f187n'], ext=('SCI', 1))
f405n_header = fits.getheader(image_filenames['f405n'], ext=('SCI', 1))
wcs_f187n = WCS(f187n_header)
wcs_f405n = WCS(f405n_header)

ext = CT06_MWLoc()


    

    

Set DATE-AVG to '2025-05-06T13:24:26.762' from MJD-AVG.
Set DATE-END to '2025-05-06T13:48:04.085' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.202551 from OBSGEO-[XYZ].
Set OBSGEO-H to 1610191901.140 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]
Set DATE-AVG to '2025-05-06T13:24:26.757' from MJD-AVG.
Set DATE-END to '2025-05-06T13:48:04.021' from MJD-END'. [astropy.wcs.wcs]
Set OBSGEO-B to   -25.202550 from OBSGEO-[XYZ].
Set OBSGEO-H to 1610191894.941 from OBSGEO-[XYZ]'. [astropy.wcs.wcs]


In [2]:
from astropy.modeling import models

keys = ['density_from_paa', 'density_from_bra', 'Q_from_density_paa', 'Q_from_density_bra', 'av_estimate', 'flux_405410', 'flux_187182', 'dered_flux_405410', 'dered_flux_187182',
'EM_from_IPaa', 'EM_from_IBra', 'I_paa', 'I_bra', 'reg_radius_au']
params = {}
for reg in reg_HII:
    

    # if ellipseskyregion, then change it to circleskyregion with radius = semi-major axis
   
    reg = CircleSkyRegion(center=reg.center, radius=reg.width/2, meta=reg.meta)
    reg_pix_405 = reg.to_pixel(wcs_f405n)
    reg_pix_187 = reg.to_pixel(wcs_f187n)
    mask_pix_405 = reg_pix_405.to_mask(mode='center',)
    mask_pix_187 = reg_pix_187.to_mask(mode='center',)

    cutout_405410 = mask_pix_405.cutout(img_for_405410.data)
    cutout_187182 = mask_pix_187.cutout(img_for_187182.data)
    mean_405410 = (np.nanmean(cutout_405410.data) * u.MJy/u.sr *
                   wcs_f405n.proj_plane_pixel_area()).to(u.Jy)
    mean_187182 = (np.nanmean(cutout_187182.data) * u.MJy/u.sr *
                   wcs_f187n.proj_plane_pixel_area()).to(u.Jy)
    area_mask_405 = np.pi * reg.radius**2
    area_mask_187 = np.pi * reg.radius**2
    area_mask_pix_405410 = area_mask_405 / wcs_f405n.proj_plane_pixel_area()
    area_mask_pix_187182 = area_mask_187 / wcs_f187n.proj_plane_pixel_area()
    flux_405410 = mean_405410 * area_mask_pix_405410
    flux_187182 = mean_187182 * area_mask_pix_187182

    flux_ratio = flux_405410 / flux_187182

    Adiff = 2.5*np.log10(3.9/flux_ratio.value) # intrinsic ratio of 3.9 is from the ratio of the emissivities of the two lines at 10^4 K, n_e=10^4 cm^-3, case B recombination (Storey & Hummer 1995)
    av_estimate = Adiff / (ext(1/1.87) - ext(1/4.05))
    print('Adiff:', Adiff, 'flux_ratio:', flux_ratio, 'av_estimate:', av_estimate)

    #print('A_V estimate for region {}: {:.2f}'.format(reg.meta['text'], av_estimate))
    dered_flux_405410 = flux_405410 * 10**(0.4 * av_estimate * ext(1/4.05))
    dered_flux_187182 = flux_187182 * 10**(0.4 * av_estimate * ext(1/1.87))
    print('flux_187182:', flux_187182, 'dered_flux_187182:', dered_flux_187182, 'flux_405410:', flux_405410, 'dered_flux_405410:', dered_flux_405410)

    tab_paa = SvoFps.get_transmission_data(f'JWST/NIRCAM.f187n')
    wave_paa = tab_paa['Wavelength'] 
    trans_paa = tab_paa['Transmission']
    # get the frequency width of PaA from svo_fps
    delta_lam = np.trapezoid(trans_paa / trans_paa.max(), wave_paa) * u.AA
    print('delta_lam:', delta_lam)
    delta_nu_paa = (const.c/1.87e-6/u.m**2 * delta_lam).to(u.Hz)
    reg_radius_au = (reg.radius.value * 5400)*u.au
    I_paa = (dered_flux_187182 * delta_nu_paa / (reg.radius)**2).to(u.erg/u.s/u.cm**2/u.sr)
    nu_paa = (const.c/(1.87e-6*u.m)).to(u.GHz)
    bb = models.BlackBody(temperature=1e4*u.K)
    EM_from_IPaa_roberto = (12.143*np.log(1/(1-I_paa/bb(nu_paa)/delta_nu_paa))*(nu_paa.to(u.GHz).value)**2.1 *(1e4)**1.35*u.pc*u.cm**(-6) ).to(u.pc*u.cm**(-6))
    density_from_paa = ((EM_from_IPaa_roberto / reg_radius_au/ 2)**0.5).to(u.cm**-3) #radius=fwhm
    print('density_from_paa:', density_from_paa)
    Q_from_density_paa = np.log10((4 * np.pi / 3 * 2.59e-13*u.cm**3/u.s * density_from_paa**2 * reg_radius_au**3).to(u.s**-1).value) 
    print('Q_from_density_paa:', Q_from_density_paa)


    tab_bra = SvoFps.get_transmission_data(f'JWST/NIRCAM.f405n')
    wave_bra = tab_bra['Wavelength']
    trans_bra = tab_bra['Transmission']
    # get the frequency width of BrA from svo_fps
    delta_lam_bra = np.trapezoid(trans_bra / trans_bra.max(), wave_bra) * u.AA
    print('delta_lam_bra:', delta_lam_bra)
    delta_nu_bra = (const.c/4.05e-6/u.m**2 * delta_lam_bra).to(u.Hz)
    I_bra = (dered_flux_405410  * delta_nu_bra / (reg.radius)**2).to(u.erg/u.s/u.cm**2/u.sr)
    nu_bra = (const.c/(4.05e-6*u.m)).to(u.GHz   )
    EM_from_IBra_roberto = (12.143*np.log(1/(1-I_bra/bb(nu_bra)/delta_nu_bra))*(nu_bra.to(u.GHz).value)**2.1 *(1e4)**1.35 *u.pc*u.cm**(-6) ).to(u.pc*u.cm**(-6))
    density_from_bra = ((EM_from_IBra_roberto / reg_radius_au/ 2)**0.5).to(u.cm**-3) #radius=fwhm
    print('density_from_bra:', density_from_bra)
    Q_from_density_bra = np.log10((4 * np.pi / 3 * 2.59e-13*u.cm**3/u.s * density_from_bra**2 * reg_radius_au**3).to(u.s**-1).value) 
    print('Q_from_density_bra:', Q_from_density_bra)


    


    inputs = [density_from_paa, density_from_bra, Q_from_density_paa, Q_from_density_bra, av_estimate, flux_405410, flux_187182, dered_flux_405410, dered_flux_187182,
    EM_from_IPaa_roberto, EM_from_IBra_roberto, I_paa, I_bra, reg_radius_au]
    for key, value in zip(keys, inputs):
        if key not in params:
            params[key] = []
        if isinstance(value, u.Quantity):
            params[key].append(value.value)
        else:
            params[key].append(value)

/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(
/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(


Adiff: 0.9269798546623997 flux_ratio: 1.6606291808995293 av_estimate: 10.885871155454765
flux_187182: 4874375.916555516 arcsec2 Jy / deg2 dered_flux_187182: 19736865.640097108 arcsec2 Jy / deg2 flux_405410: 8094530.88570598 arcsec2 Jy / deg2 dered_flux_405410: 13955934.032443525 arcsec2 Jy / deg2
delta_lam: 236.59542237910327 Angstrom
density_from_paa: 4168.393175439592 1 / cm3
Q_from_density_paa: 49.544955145287
delta_lam_bra: 454.94571260238206 Angstrom
density_from_bra: 3011.4384418607638 1 / cm3
Q_from_density_bra: 49.26256577288955
Adiff: 0.8206166530374561 flux_ratio: 1.8315464858185007 av_estimate: 9.636808295299646
flux_187182: 564136.4630273909 arcsec2 Jy / deg2 dered_flux_187182: 1945601.7198006208 arcsec2 Jy / deg2 flux_405410: 1033242.1563798963 arcsec2 Jy / deg2 dered_flux_405410: 1673498.1094847366 arcsec2 Jy / deg2
delta_lam: 236.59542237910327 Angstrom
density_from_paa: 11291.139650922347 1 / cm3
Q_from_density_paa: 48.53873119290178
delta_lam_bra: 454.94571260238206 An

/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(


Adiff: 0.19586195568311604 flux_ratio: 3.2562656262596557 av_estimate: 2.3000802046537463
flux_187182: 229689.59599325116 arcsec2 Jy / deg2 dered_flux_187182: 308652.9927471798 arcsec2 Jy / deg2 flux_405410: 747930.3361422913 arcsec2 Jy / deg2 dered_flux_405410: 839161.4694719841 arcsec2 Jy / deg2
delta_lam: 236.59542237910327 Angstrom
density_from_paa: 7597.695977358539 1 / cm3
Q_from_density_paa: 47.73914749187461
delta_lam_bra: 454.94571260238206 Angstrom
density_from_bra: 10763.027653148776 1 / cm3
Q_from_density_bra: 48.04165258162054


/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(


...

/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(
/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(


Adiff: -0.3109856350961982 flux_ratio: 5.193484754258972 av_estimate: -3.652020632193143
flux_187182: 91317106.37933971 arcsec2 Jy / deg2 dered_flux_187182: 57120722.7715615 arcsec2 Jy / deg2 flux_405410: 474253999.78414553 arcsec2 Jy / deg2 dered_flux_405410: 395045218.14152163 arcsec2 Jy / deg2
delta_lam: 236.59542237910327 Angstrom
density_from_paa: 1414.9901005130173 1 / cm3
Q_from_density_paa: 50.006474526377836
delta_lam_bra: 454.94571260238206 Angstrom
density_from_bra: 3197.0029253703324 1 / cm3
Q_from_density_bra: 50.71445379075034
Adiff: -0.6332175752696861 flux_ratio: 6.987979739294119 av_estimate: -7.43611083141145
flux_187182: 202073.1751422264 arcsec2 Jy / deg2 dered_flux_187182: 77736.17514896874 arcsec2 Jy / deg2 flux_405410: 1412083.25374871 arcsec2 Jy / deg2 dered_flux_405410: 973333.8684252296 arcsec2 Jy / deg2
delta_lam: 236.59542237910327 Angstrom
density_from_paa: 2179.6457307811165 1 / cm3
Q_from_density_paa: 47.14030650871945
delta_lam_bra: 454.94571260238206 An

/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(


Adiff: 0.4673721942501274 flux_ratio: 2.535804273028921 av_estimate: 5.48852649025689
flux_187182: 29772645.157747045 arcsec2 Jy / deg2 dered_flux_187182: 60261602.09731558 arcsec2 Jy / deg2 flux_405410: 75497600.81038877 arcsec2 Jy / deg2 dered_flux_405410: 99359071.66647854 arcsec2 Jy / deg2
delta_lam: 236.59542237910327 Angstrom
density_from_paa: 2262.714347545119 1 / cm3
Q_from_density_paa: 50.02971739160053
delta_lam_bra: 454.94571260238206 Angstrom
density_from_bra: 2496.194300819671 1 / cm3
Q_from_density_bra: 50.11501470487244
Adiff: -0.15537459673444134 flux_ratio: 4.50002001814873 av_estimate: -1.824622004863163
flux_187182: 50440180.46709622 arcsec2 Jy / deg2 dered_flux_187182: 39900181.08185331 arcsec2 Jy / deg2 flux_405410: 226981821.82096753 arcsec2 Jy / deg2 dered_flux_405410: 207175860.37778252 arcsec2 Jy / deg2
delta_lam: 236.59542237910327 Angstrom
density_from_paa: 5554.2733180441965 1 / cm3
Q_from_density_paa: 49.85065244643162
delta_lam_bra: 454.94571260238206 Angs

/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(
/blue/adamginsburg/t.yoo/from_red/miniconda3/envs/py311/lib/python3.11/site-packages/dust_extinction/helpers.py:12: SpectralUnitsWarning: x has no units, assuming x units are inverse microns
  warnings.warn(
